# 101_05 — PSBPM-FD con cuatro predictores y tres intervalos contra FAR y RF · Metro Interstate (I-94) horario · volumen de tráfico · rezago 1 · $M = 10$

Paso **3c**, posterior a `101_04_evaluacion`, con el pipeline de las 30x sobre datos reales. §3.3 lee los
cuatro predictores puntuales del PSBPM-FD y sus tres construcciones de intervalo, una fila
`PSBPM-FD (<predictor>)` por cada uno; §4, §6 y §8 los ponen junto a FAR y RF (§6 cada predictor contra
FAR y RF por separado); §5 y §7 traen el FAR y las tres construcciones del PSBPM-FD (equi-colas, banda
simultánea y región de máxima densidad, con la esperanza como curva asociada; `111_*` en el barrido);
§9 y §10 van sobre la ventana de detalle que fijó el `_04`.

Los cálculos viven en `fit/comparacion_barrido.py` y las figuras en `graphics/viz_comparacion.py`;
este notebook los orquesta. La predicción y las bandas del PSBPM-FD se **leen** de
`banda_funcional_psbp.npz` (lo persiste `101_04`): no se recalcula nada desde las trazas.

| Sección | Qué responde |
|---|---|
| 1 | `[CONFIG]`, artefactos y diseño común |
| 2 | validaciones: mismo corte, mismas covariables, mismo objetivo para todos |
| 3 | los competidores: FAR(1), Random Forest (rezago 1 cruzado) y el PSBPM-FD |
| 4 | **Bloque A** (MAE, RMSE, error máximo) — los tres modelos |
| 5 | **Bloque B** (Winkler, cobertura simultánea, PICP, MPIW) — FAR y las tres construcciones del PSBPM-FD |
| 6 | Bloque A: % de ventanas ganadas y cuánto mejor es cada modelo |
| 7 | Bloque B: lo mismo, sobre las bandas |
| 8 | resumen: mínimo, máximo y promedio por ventana, y las métricas contra $M$ |
| 9 | la ventana de detalle con las bandas superpuestas |
| 10 | comportamientos predictivos contra lo observado |

> **Corrida 101 — la 31 reevaluada.** Covariables cruzadas con el rezago 1 ($p = M = 10$), `M = 10`, 1 cadena,
> `chungEscG` con $a_\nu = 2$. Representación B-spline y FPCA estimadas en train. El objetivo de
> evaluación es la curva suavizada; no hay curva verdadera. Referencias, las de la 31: el **FAR(1)**
> (`p = N_LAGS = 1`) y el Random Forest con las mismas covariables del PSBPM-FD (rezago 1 cruzado).
> Las filas esperanza / equi-colas, FAR y RF deben reproducir al bit las de la 31
> (`reports/reales/real_trafico_m10_v01_barrido_M/98_*`, `99_*`). La 102 (la 32, rezagos $\{1, 7\}$) es el contraste.


## 1. Imports, `[CONFIG]` y artefactos

In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
import json

import numpy as np

from model_psbp_fd.fit.convergencia_barrido import cargar_artefactos
from model_psbp_fd.fit.evaluacion_barrido import (
    METRICAS_A as _MA, METRICAS_B as _MB, cargar_representaciones, serie_origenes,
)
from model_psbp_fd.fit.comparacion_barrido import (
    G_FAR, G_PSG, G_PSB, PISO,
    preparar_diseno, ajustar_far, ajustar_rf, cargar_psbp,
    tablas_modelos, verificar_cadena_modelos,
    construir_bandas, tabla_bloque_B, ventana_bloque_B,
    ganadores_por_ventana, resumen_min_max, promedio_vs_M,
    origenes_extremos, contraste_origenes, nombre_psbp, validar_comportamientos_barrido,
    G_PSS, G_PSH, bloqueB_contra_far,
)
from model_psbp_fd.fit.evaluacion_barrido import INTERVALOS as INTERVALOS_TODOS
from model_psbp_fd.fit.resumenes_predictiva import PREDICTORES as PREDICTORES_TODOS
from model_psbp_fd.graphics import (
    plot_ventana_movil, ESTILOS_MODELOS, ESTILOS_MODELOS_PREDICTORES, plot_validacion_observado,
    plot_ganador_modelo, plot_metricas_vs_M, plot_bandas_contraste,
)
from model_psbp_fd.fit.comportamientos import cargar_comportamientos, validar_observado
from model_psbp_fd.fit.intervalos_predictiva import ancho_tramos, cobertura_tramos
from model_psbp_fd.utils import get_project_root, experiment_id, normalizar_lista_M
from model_psbp_fd.utils.rutas import ruta_barrido_M

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

### 1.1 `[CONFIG]` Identificación del experimento y barrido en $M$

In [ ]:
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Identidad de la corrida: mismos valores que 101_01 §1.1. El ID es
# real_<serie>_v<NN>_m<NN>; las funciones de fit/ lo arman con (BASE_LIB, ESCENARIO_ID, None).
BASENAME, SERIE_ID, VENTANA_ID = "real", "trafico_101", 1   # ventana 1: 2015-07-01 .. 2018-09-30, 70/30 (31_01 §1.1; artefactos copiados por 00_copiar_31.py)
M_FPCA_LIST = (10,)
DOMINIO      = "reales"
BASE_LIB     = f"{BASENAME}_{SERIE_ID}"
ESCENARIO_ID = f"v{VENTANA_ID:02d}"
REPLICA_ID   = None

# Propias de la comparación. Ventanas, nivel y objetivo se LEEN de eval_config.json
# (artefacto de 101_01): no se redeclaran aquí.
PESOS_TAU = None          # cuadratura trapezoidal por defecto
PREFIJO   = 70            # este notebook escribe 70-83 en cada M y 96-99, 105-107, 111 en el barrido
INTERVALOS  = INTERVALOS_TODOS    # equicolas, simultanea, hdr: una banda del PSBPM-FD por construcción
N_DETALLE   = 10        # orígenes consecutivos de prueba para el análisis predictivo (§9, §10)
T0_DETALLE  = None      # el _05 LEE la ventana del _04 (112_ventana_detalle.json); un entero sólo la verifica
PREDICTORES = PREDICTORES_TODOS   # una fila del PSBPM-FD por predictor puntual (101_04 §3)

# FAR (§3.1): FAR(1), p = N_LAGS = 1 fijo, como en la 31. Tope del barrido de kn
# (acota en K = 10, no en L) y criterio de far.cv.
KN_MAX      = 12
CRITERIO_KN = "L2"

# Random Forest (§3.2): hiperparámetros fijos entre escenarios y entre puntos del barrido.
SEED_ML   = 20260830
PARAMS_RF = dict(n_estimators=500, min_samples_leaf=5, max_features="sqrt",
                 random_state=SEED_ML, n_jobs=-1)

# Bloque B (§5): FAR con banda gaussiana de residuos de train y PSBPM-FD con su banda
# nativa (cuantiles de su predictiva).
IC_POR_TAU                = True    # sigma_eps(tau) punto a punto (False = ancho constante)
IC_CORRECCION_ESTIMACION  = True    # infla sigma por sqrt(1 + kn/n_train)
IC_PSBP_TAMBIEN_GAUSSIANA = False   # control opcional: no se corre (sólo FAR gaussiano y PSBPM-FD nativo)

# §9: por qué métrica se eligen los orígenes extremos, cuántos, qué dos bandas se
# contrastan (prefijos, en orden) y en qué M (None = todos).
METRICA_SEL_IC     = "rmse_f"     # una de METRICAS_A_COLS
N_EXTREMOS         = 5
BANDA_A_CONTRASTAR = ("FAR", "PSBPM-FD (predictiva")
M_DETALLE          = None

# Las 10 métricas del marco teórico 02_02_03, en su orden. No se agregan otras.
METRICAS_A = [("1. MAE  (L^1)", "mae_f"), ("2. RMSE (L^2)", "rmse_f"),
              ("3. E_max promedio por curva", "linf_medio"),
              ("4. E_max peor de la ventana", "linf_max")]
METRICAS_B = [("5. MPIW", "mpiw"), ("6. PICP puntual", "picp"),
              ("7. PICPB curva completa", "picp_simultaneo"), ("8. Winkler", "winkler"),
              ("9. Winkler max promedio", "winkler_max_medio"),
              ("10. Winkler max peor ventana", "winkler_max_glob")]
assert METRICAS_A == _MA and METRICAS_B == _MB, "METRICAS_* difieren de fit.evaluacion_barrido."
METRICAS_A_COLS = [c for _, c in METRICAS_A]
METRICAS_B_COLS = [c for _, c in METRICAS_B]
assert METRICA_SEL_IC in METRICAS_A_COLS[:3], "METRICA_SEL_IC: mae_f, rmse_f o linf_medio."

M_FPCA_LIST     = normalizar_lista_M(M_FPCA_LIST)
EXPERIMENT_BASE = experiment_id(BASE_LIB, ESCENARIO_ID, REPLICA_ID)
PATH_BARRIDO    = ruta_barrido_M(BASE_LIB, ESCENARIO_ID, REPLICA_ID, dominio=DOMINIO,
                                 project_root=PROJECT_ROOT)

print(f"EXPERIMENT_BASE : {EXPERIMENT_BASE}")
print(f"barrido en M    : {list(M_FPCA_LIST)}")
print(f"salidas cruzadas: {PATH_BARRIDO}   (archivos 96-99, 105-107)")

In [ ]:
# EST[M]: estado de cada punto (cruza el contrato al cargar). DIS: diseño común.
EST, saltados = cargar_artefactos(BASE_LIB, ESCENARIO_ID, REPLICA_ID, M_FPCA_LIST,
                                  PROJECT_ROOT, con_test=True, dominio=DOMINIO)
DIS  = cargar_representaciones(EST)
ORIG = serie_origenes(DIS)

T0, N_LAGS, NIVEL = DIS["T0"], DIS["N_LAGS"], DIS["NIVEL"]
VENTANAS_W, W_REF = DIS["VENTANAS_W"], DIS["W_REF"]
grilla, X_obj_ev  = DIS["grilla"], ORIG["X_obj_ev"]
t_orig, es_train  = ORIG["t_orig"], ORIG["es_train"]

In [ ]:
# Ventana de detalle: la que eligió el _04 (§3.2), leída de 112_ventana_detalle.json; no se recalcula.
VD = json.loads((PATH_BARRIDO / "112_ventana_detalle.json").read_text(encoding="utf-8"))
assert VD["n"] == N_DETALLE, f"el _04 usó N_DETALLE={VD['n']}"
assert T0_DETALLE is None or int(T0_DETALLE) == VD["t0"], f"el _04 usó T0_DETALLE={VD['t0']}"
T0_DET   = int(VD["t0"])
IDX_DET  = np.flatnonzero((t_orig >= T0_DET) & (t_orig < T0_DET + N_DETALLE))
assert len(IDX_DET) == N_DETALLE and not es_train[IDX_DET].any()
MASK_DET = np.isin(np.arange(len(t_orig)), IDX_DET)
T_DET    = t_orig[IDX_DET]
print(f"Ventana de detalle del _04: t = {T0_DET}..{T0_DET + N_DETALLE - 1}   ·   {VD['criterio']}: {VD['valor']}")

## 2. Validaciones para aplicar los modelos

In [ ]:
preparar_diseno(EST, DIS, ORIG)
# Rezagos efectivos del diseño (manifest["lags"]); N_LAGS = max(lags) es el desplazamiento y el orden del FAR completo.
LAGS = tuple(int(l) for l in EST[next(iter(EST))]["manifest"].get("lags", range(1, N_LAGS + 1)))
print(f"rezagos efectivos de las covariables: {LAGS}   ·   N_LAGS = {N_LAGS} (desplazamiento; FAR({N_LAGS}))")


## 3. Los competidores

### 3.1 FAR(1), $p = N_{lags} = 1$, sobre los COEFICIENTES de la base B-spline, en la métrica $L^2$

Se ajusta sobre los coeficientes de la representación B-spline ($K$ funciones, no truncada a $M$), no
sobre la curva en la grilla. `p = N_LAGS = 1` fijo, como en la 31: seleccionarlo daría una ventaja de
especificación que el PSBPM-FD no tiene. La base B-spline **no** es ortonormal, así que el blanqueo con
la Cholesky de la Gram $W$ es lo que hace que la estimación sea en $L^2$ y no en la métrica euclídea de
los coeficientes; la dimensión $k_n$ se elige por predicción a un paso sobre el último 20 % del
entrenamiento.


In [ ]:
FAR = ajustar_far(EST, DIS, ORIG, KN_MAX, CRITERIO_KN)

Sin validación cruzada sobre el bloque de prueba y con hiperparámetros fijos. Cada componente se ajusta
con **las mismas covariables cruzadas que el PSBPM-FD**, el rezago 1 de los $M$ scores, para que la
comparación sea de modelo y no de insumos. Con `LAGS = (1,)` el barrido de órdenes (`ordenes=LAGS`)
tiene un solo candidato, el de la 31.


In [ ]:
ajustar_rf(EST, DIS, ORIG, PARAMS_RF, PESOS_TAU, ordenes=LAGS)

### 3.3 PSBPM-FD

Sus cuatro predictores puntuales (esperanza analítica, mediana puntual, medoide $L^1$ y átomo
modal, docs §03_05_04) y sus tres construcciones de intervalo (equi-colas, banda simultánea y
región de máxima densidad, §03_05_05) se **leen** de `banda_funcional_psbp.npz`, que persiste
`101_04`: una fila `PSBPM-FD (<predictor>)` por predictor en el Bloque A y una banda por
construcción en el Bloque B, con la esperanza como curva asociada. Los $M$ sin artefacto se
quitan con aviso.

In [ ]:
M_OK = cargar_psbp(EST, DIS, ORIG, predictores=PREDICTORES, intervalos=INTERVALOS)

## 4. Bloque A: error puntual, todos los modelos

Las **mismas tres normas** que `200_04`, sobre la misma ventana móvil y contra
la misma curva verdadera:

| | qué estima |
|---|---|
| **MAE** ($L^1$) | mediana condicional, robusta |
| **RMSE** ($L^2$) | media condicional |
| **Error máximo** ($L^\infty$, promedio por curva) | peor caso: por curva $\max_\tau\lvert e_t(\tau)\rvert$, promediado sobre la ventana |

Aplican a **todos** los competidores: FAR, RF y PSBPM-FD. No requieren predictiva, sólo una predicción puntual.

In [ ]:
for M in M_OK:
    tablas_modelos(EST[M], DIS, ORIG, PESOS_TAU, PREFIJO)
verificar_cadena_modelos(EST, W_REF)

# Una figura por M; el piso de truncamiento va en las tablas pero no en la figura.
for M in M_OK:
    e = EST[M]
    tw = e["tablas_w"][W_REF]
    tf = tw[tw["modelo"] != PISO].copy()
    tf.attrs["w"] = W_REF
    plot_ventana_movil(
        tf, T0, METRICAS_A_COLS, columna_grupo="modelo", estilos=ESTILOS_MODELOS_PREDICTORES,
        title=f"M={M} · Bloque A por modelo (w={W_REF}) · MAE, RMSE, error máximo · contra la curva suavizada",
        save_path=str(e["paths"]["out_report"] / f"{PREFIJO + 8}_ventana_bloqueA_modelos_w{W_REF}.png"),
        verbose=False)
    plt.show()

## 5. Bloque B: intervalos, sólo FAR y PSBPM-FD (tres construcciones)

Las **mismas cuatro cifras** que `200_04` —Winkler, cobertura simultánea,
cobertura puntual y MPIW— sobre la misma ventana móvil.

La diferencia con el Bloque A es a qué modelos aplica. El FAR **no** es un
algoritmo de predicción puntual: es un modelo, $X_{t+1}=\rho(X_t)+\varepsilon$,
de modo que su intervalo a un paso sale de su propia especificación y no de un
procedimiento adosado. El PSBPM-FD tiene su predictiva muestreada. El RF no
es un modelo de probabilidad: rellenarle una banda
gaussiana de residuos dentro de muestra lo haría parecer artificialmente
angostas y decidiría la comparación por el supuesto, no por el método.

In [ ]:
for M in M_OK:
    e = EST[M]
    construir_bandas(e, DIS, ORIG, IC_POR_TAU, IC_CORRECCION_ESTIMACION, IC_PSBP_TAMBIEN_GAUSSIANA,
                     intervalos_psbp=INTERVALOS)
    B_df = tabla_bloque_B(e, DIS, ORIG)
    display(B_df.loc["test"].style
            .format({"winkler": "{:.4f}", "picp": "{:.4f}", "ee_picp": "{:.4f}", "ace": "{:+.4f}",
                     "mpiw": "{:.4f}", "picp_simultaneo": "{:.4f}"})
            .background_gradient(subset=["winkler"], cmap="RdYlGn_r")
            .set_caption(f"M={M} · Bloque B, bloque de PRUEBA · PRIMARIA: winkler · nivel nominal {NIVEL:.0%}"))

In [ ]:
# Winkler sobre la ventana móvil y su descomposición: dice CUÁNDO falla la banda, no sólo cuánto.
for M in M_OK:
    e = EST[M]
    vb = ventana_bloque_B(e, DIS, ORIG, PESOS_TAU, PREFIJO)
    plot_ventana_movil(
        vb, T0, ["winkler", "picp_simultaneo", "picp", "mpiw"], columna_grupo="banda",
        title=f"M={M} · Bloque B sobre la ventana móvil (w={W_REF}) — winkler es la primaria; el resto lo descompone",
        save_path=str(e["paths"]["out_report"] / f"81_ventana_bloqueB_w{W_REF}.png"), verbose=False)
    plt.show()

## 6. Bloque A: quién gana cada ventana, y por cuánto

Dos lecturas distintas y complementarias sobre la misma tabla de ventana:

1. **% de ventanas ganadas** — para cada ventana se marca el modelo con el
   valor más bajo de la métrica. Es una lectura más dura que el promedio: un
   modelo puede tener mejor promedio y ganar pocas ventanas si pierde por poco
   muchas veces y gana por mucho unas pocas.
2. **Cuánto mejor** — la razón de cada modelo contra el mejor de la ventana y
   contra el FAR, la referencia lineal (aquí el FAR(1), con la misma
   información que el PSBPM-FD).

Ambas se calculan sobre las ventanas que **no cruzan $T_0$**.

In [ ]:
gana_A, rel_A = ganadores_por_ventana(EST, "ventana_df", "modelo", METRICAS_A, NIVEL, excluir=(PISO,))
gana_A.to_csv(PATH_BARRIDO / "96_ganador_ventana_bloqueA.csv", index=False)
rel_A.to_csv(PATH_BARRIDO / "96_razones_bloqueA.csv", index=False)

for etiqueta, _col in METRICAS_A:
    for tabla, v, fmt, cmap, cap in (
            (gana_A, "pct_ventanas_ganadas", "{:.1%}", "RdYlGn",
             f"{etiqueta} · % de ventanas de PRUEBA ganadas (w={W_REF}, sin cruzar T0)"),
            (rel_A, "razon_vs_mejor", "{:.3f}", "RdYlGn_r",
             f"{etiqueta} · razón contra el MEJOR de cada ventana (test) · 1.000 = gana siempre")):
        piv = (tabla[(tabla.metrica == etiqueta) & (tabla.bloque == "test")]
               .pivot(index="modelo", columns="M", values=v))
        display(piv.style.format(fmt, na_rep="-").background_gradient(cmap=cmap, axis=None).set_caption(cap))

for M in M_OK:
    plot_ganador_modelo(gana_A, M, METRICAS_A, W_REF,
                        EST[M]["paths"]["out_report"] / f"{PREFIJO + 8}_ganador_ventana_bloqueA.png")
    plt.show()

In [ ]:
# Cada predictor del PSBPM-FD contra FAR y RF por separado (106_ganador_psbp_por_predictor.csv).
partes = []
for p in PREDICTORES:
    otros = [nombre_psbp(q) for q in PREDICTORES if q != p]
    g, _ = ganadores_por_ventana(EST, "ventana_df", "modelo", METRICAS_A, NIVEL, excluir=(PISO, *otros))
    partes.append(g[g.modelo == nombre_psbp(p)].assign(predictor=p))
gana_P = pd.concat(partes, ignore_index=True)
gana_P.to_csv(PATH_BARRIDO / "106_ganador_psbp_por_predictor.csv", index=False)

for etiqueta, _c in METRICAS_A:
    display(gana_P[(gana_P.metrica == etiqueta) & (gana_P.bloque == "test")]
            .pivot(index="predictor", columns="M", values="pct_ventanas_ganadas").loc[list(PREDICTORES)]
            .style.format("{:.1%}").background_gradient(cmap="RdYlGn", axis=None)
            .set_caption(f"{etiqueta} · % de ventanas TEST que gana el PSBPM-FD con cada predictor, contra FAR y RF"))

## 7. Bloque B: quién gana cada ventana, y por cuánto

La misma lectura de §6, restringida a las series que **tienen un mecanismo de
intervalo propio**: el FAR —cuyo IC a un paso sale de su propia especificación
$X_{t+1}=\rho(X_t)+\varepsilon$— y el PSBPM-FD —cuya predictiva se muestrea y
cuyos cuantiles empíricos dan la banda—.

El Winkler es la única regla de puntuación propia del bloque, así que es la
que decide quién gana; PICP y MPIW lo descomponen en cobertura y ancho.

In [ ]:
# En picp / picp_simultaneo "ganar" es estar más cerca del nominal, no tener el valor más bajo.
gana_B, rel_B = ganadores_por_ventana(EST, "ventana_B", "banda", METRICAS_B, NIVEL)
gana_B.to_csv(PATH_BARRIDO / "97_ganador_ventana_bloqueB.csv", index=False)
rel_B.to_csv(PATH_BARRIDO / "97_razones_bloqueB.csv", index=False)

for etiqueta, col in METRICAS_B:
    piv = (gana_B[(gana_B.metrica == etiqueta) & (gana_B.bloque == "test")]
           .pivot(index="banda", columns="M", values="pct_ventanas_ganadas"))
    crit = "más cerca del nominal" if col.startswith("picp") else "menor es mejor"
    display(piv.style.format("{:.1%}", na_rep="-").background_gradient(cmap="RdYlGn", axis=None)
            .set_caption(f"{etiqueta} · % de ventanas de PRUEBA ganadas (criterio: {crit})"))

## 8. Cuadro resumen: mínimo, máximo y promedio

Igual que en `200_04`: la ventana móvil genera una **serie** por métrica y por
modelo, no una cifra. El resumen muestra los tres números que describen esa
serie —mínimo, máximo y promedio— por bloque train/test, excluyendo las
ventanas que cruzan $T_0$.

El **máximo** es el que no aparece en ninguna tabla de promedios y suele ser el
que decide: un modelo con buen promedio y un máximo muy alto falla feo en
algún tramo, y eso es información distinta de su error típico.

In [ ]:
resumen_A = resumen_min_max(EST, "ventana_df", "modelo", METRICAS_A, excluir=(PISO,))
resumen_B = resumen_min_max(EST, "ventana_B", "banda", METRICAS_B)
resumen_A.to_csv(PATH_BARRIDO / "98_resumen_bloqueA_min_max_prom.csv", index=False)
resumen_B.to_csv(PATH_BARRIDO / "99_resumen_bloqueB_min_max_prom.csv", index=False)

for M in M_OK:
    for res, grupo, fmt, nombre in ((resumen_A, "modelo", "{:.6f}", "Bloque A"), (resumen_B, "banda", "{:.4f}", "Bloque B")):
        sub = (res[(res.M == M) & (res.bloque == "test")]
               .set_index(["metrica", grupo])[["minimo", "maximo", "promedio"]])
        display(sub.style.format(fmt).background_gradient(subset=["maximo"], cmap="RdYlGn_r")
                .set_caption(f"M={M} · {nombre} en el TEST · mínimo / máximo / promedio sobre las ventanas (w={W_REF})"))

In [ ]:
# Las métricas contra M: promedio en test sobre las ventanas, una línea por modelo / banda.
ESTILOS_BANDAS = {G_FAR: dict(color="#1f6f8b"), G_PSG: dict(color="#e08e0b"), G_PSB: dict(color="#c0392b"),
                  G_PSS: dict(color="#8856a7"), G_PSH: dict(color="#2ca25f")}

plot_metricas_vs_M(promedio_vs_M(resumen_A, "modelo"), METRICAS_A, ESTILOS_MODELOS_PREDICTORES,
                   f"{EXPERIMENT_BASE} · Bloque A en test vs M (promedio sobre ventanas, w={W_REF})",
                   save_path=PATH_BARRIDO / f"{PREFIJO + 6}_bloqueA_vs_M.png")
plt.show()
plot_metricas_vs_M(promedio_vs_M(resumen_B, "banda"),
                   [("Winkler", "winkler"), ("PICP", "picp"), ("PICPB", "picp_simultaneo"), ("MPIW", "mpiw")],
                   ESTILOS_BANDAS,
                   f"{EXPERIMENT_BASE} · Bloque B en test vs M (promedio sobre ventanas, w={W_REF})",
                   nominal=NIVEL, save_path=PATH_BARRIDO / f"{PREFIJO + 7}_bloqueB_vs_M.png")
plt.show()

### 8.1 Cada construcción del PSBPM-FD contra el FAR

Por $M$: Winkler medio en test de cada construcción menos el del FAR (negativo = mejor que el
FAR) y % de ventanas de test en que le gana cara a cara (`111_bloqueB_intervalos_vs_FAR.csv`).

In [ ]:
B_VS_FAR = bloqueB_contra_far(EST, PATH_BARRIDO)
display(B_VS_FAR.style.format(precision=4)
        .set_caption("Bloque B · construcciones del PSBPM-FD contra el FAR (test, w de referencia)"))

## 9. Intervalos de credibilidad: FAR contra PSBPM-FD, en la ventana de detalle

Los promedios de §7 y §8 dicen *cuánto*, no *dónde*. Aquí se eligen los
$N$ orígenes de prueba **mejor** y **peor** predichos según
`METRICA_SEL_IC` ([CONFIG]) y se dibujan las dos bandas superpuestas sobre la
misma curva verdadera.

Es donde se ve la diferencia cualitativa que las cifras agregadas promedian: el
IC del FAR tiene **ancho que no depende del origen** —su $\Sigma_\varepsilon$
es la misma para todo $t$, de modo que sólo hereda la forma de $\tau$— mientras
que la banda del PSBPM-FD sale de los cuantiles de una predictiva que sí cambia
con el estado rezagado. En la 31 la predictiva del PSBPM-FD era bimodal el 70 % de los días (laborable
contra fin de semana), y es aquí donde se ve cómo cada construcción (equi-colas, HDR en tramos y
simultánea) la recoge y si la banda cambia de ancho y de forma con el día de la semana mientras la del
FAR no puede.

Los orígenes son los `N_DETALLE` de la ventana de detalle del `_04` (§3.2), no los mejores y peores del test: tres figuras por $M$ (FAR contra la equi-colas, contra la región de máxima densidad en tramos y contra la banda simultánea) y la tabla de ancho y cobertura por origen, con la cobertura de curva completa como cuenta.

In [ ]:
M_9 = M_OK if M_DETALLE is None else tuple(m for m in M_DETALLE if m in M_OK)
DETALLE = {"detalle": IDX_DET}
for M in M_9:
    for par, sufijo in ((BANDA_A_CONTRASTAR, ""), (("FAR", G_PSH), "_hdr"), (("FAR", G_PSS), "_simultanea")):
        plot_bandas_contraste(
            EST[M]["BANDAS"], DETALLE, X_obj_ev, grilla, t_orig, M, METRICA_SEL_IC, par, "detalle",
            EST[M]["paths"]["out_report"] / f"{PREFIJO + 9}_ic_contraste_detalle{sufijo}.png")
        plt.show()

In [ ]:
# Ancho y cobertura por origen de cada banda en la ventana de detalle (79_ic_contraste_origenes.csv por M).
filas = []
for M in M_9:
    for i in IDX_DET:
        for nombre, ((L, U), _Xp) in EST[M]["BANDAS"].items():
            dentro = cobertura_tramos(X_obj_ev[i], L[i], U[i])
            filas.append({"M": M, "t": int(t_orig[i]), "banda": nombre,
                          "mpiw": float(ancho_tramos(L[i], U[i]).mean()),
                          "picp_puntual": float(dentro.mean()), "I_t_simultaneo": float(dentro.all())})
ic_df = pd.DataFrame(filas)
for M in M_9:
    ic_df[ic_df.M == M].drop(columns="M").to_csv(
        EST[M]["paths"]["out_report"] / f"{PREFIJO + 9}_ic_contraste_origenes.csv", index=False)
display(ic_df.groupby(["M", "banda"]).agg(mpiw=("mpiw", "mean"), picp_puntual=("picp_puntual", "mean"),
                                          curvas_cubiertas=("I_t_simultaneo", "sum"))
        .style.format({"mpiw": "{:.3f}", "picp_puntual": "{:.3f}", "curvas_cubiertas": "{:.0f}"})
        .set_caption(f"Ventana de detalle (t={T0_DET}..): curvas_cubiertas = cuántas de las {N_DETALLE} "
                     "quedan enteras dentro de la banda (descriptivo)"))
print("Lectura: si el MPIW del FAR es prácticamente el mismo en todos los orígenes, es porque su ancho NO")
print("depende del origen (su Sigma_eps es la misma para todo t); que el del PSBPM-FD cambie entre orígenes")
print("es la flexibilidad de una predictiva que depende del estado.")

## 10. Comportamientos predictivos contra lo observado

Sobre los `N_DETALLE` orígenes de la ventana de detalle (§3.2 del `_04`), como **cuentas** (k de 10, descriptivas). Lee `comportamientos_psbp.npz` (`101_04` §12). Por origen: el comportamiento en que cae lo
observado (centroide más cercano en $L^2$), su probabilidad contra la esperada bajo
calibración ($\sum_c p_c^2$), si es el de mayor masa, y cobertura, ancho y Winkler de tres
bandas:

| banda | qué es |
|---|---|
| **real** | la condicional del comportamiento realizado, elegida mirando lo observado: cota de lo que daría saber el comportamiento |
| **mp** | la condicional del más probable: un pronóstico |
| **marg** | la marginal, la del Bloque B |

En los orígenes con $K = 1$ las tres coinciden, por eso el resumen separa $K \ge 2$. Se
reporta contra los dos objetivos; la banda vive en las $M$ autofunciones, así que el que
corresponde es $X_t^{(M)}$ (`representacion_fpca`).

In [ ]:
# validar_observado sobre la ventana de detalle: la máscara de los comportamientos se restringe a ella.
partes, res = [], []
for M in M_OK:
    e = EST[M]
    C = cargar_comportamientos(e["paths"])
    assert C is not None and np.array_equal(C["t_orig"], t_orig), f"[M={M}] falta el §12 del _04."
    C["mascara"] = MASK_DET.copy()
    li_f, ls_f = e["BANDA_PSBP"]
    por_M = []
    for objetivo, Y in (("curva_suavizada", X_obj_ev), ("representacion_fpca", e["X_proj"])):
        # Error en el origen de la curva típica (realizado = cota, más probable = pronóstico) y de las
        # referencias puntuales en los mismos orígenes: esperanza del PSBPM-FD, FAR y RF.
        v = validar_observado(C, e["Y_obs"], Y, li_f, ls_f, grilla, PESOS_TAU,
                              curvas_ref={"esperanza": e["X_PSBP"], "FAR": e["CURVAS"]["FAR"],
                                          "RF": e["CURVAS"]["RF"]})
        v.insert(0, "objetivo", objetivo)
        v.insert(1, "t", t_orig[v["i"].to_numpy()])
        por_M.append(v)
        res.append({"M": M, "objetivo": objetivo, "origenes": len(v), "K >= 2": int((v.K >= 2).sum()),
                    "cae en el más probable": int(v.acierto.sum()),
                    "p_obs media": float(v.p_obs.mean()), "p_esperada media": float(v.p_esperada.mean()),
                    **{f"{m}_{k}": float(v[f"{m}_{k}"].mean()) for m in ("mae", "rmse")
                       for k in ("real", "mp", "esperanza", "FAR", "RF")},
                    "mp gana a FAR": int((v.mae_mp < v.mae_FAR).sum()),
                    "mp gana a RF": int((v.mae_mp < v.mae_RF).sum()),
                    "mp gana a esperanza": int((v.mae_mp < v.mae_esperanza).sum()),
                    "curva entera en banda real": int(v.picpb_real.sum()),
                    "curva entera en banda mp": int(v.picpb_mp.sum()),
                    "curva entera en banda marginal": int(v.picpb_marg.sum()),
                    "mpiw_real": float(v.mpiw_real.mean()), "mpiw_mp": float(v.mpiw_mp.mean()),
                    "mpiw_marg": float(v.mpiw_marg.mean()), "winkler_real": float(v.winkler_real.mean()),
                    "winkler_mp": float(v.winkler_mp.mean()), "winkler_marg": float(v.winkler_marg.mean())})
    pd.concat(por_M, ignore_index=True).to_csv(e["paths"]["out_report"] / "82_validacion_comportamientos.csv",
                                              index=False)
    pd.DataFrame([r for r in res if r["M"] == M]).to_csv(e["paths"]["out_report"] / "83_validacion_resumen.csv",
                                                         index=False)
VAL_COMP = pd.DataFrame(res).assign(t0_detalle=T0_DET)
VAL_COMP.to_csv(PATH_BARRIDO / "105_validacion_comportamientos_por_M.csv", index=False)
for objetivo in ("representacion_fpca", "curva_suavizada"):
    display(VAL_COMP[VAL_COMP.objetivo == objetivo].drop(columns=["objetivo", "t0_detalle"]).set_index("M")
            .style.format(precision=3)
            .set_caption(f"Ventana de detalle (t={T0_DET}..{T0_DET + N_DETALLE - 1}) · objetivo = {objetivo} · "
                         f"cuentas sobre {N_DETALLE} orígenes, descriptivas"))